# Keystroke Dynamics Offline Benchmark

**Dataset**: CMU Keystroke Dynamics Benchmark (`DSL-StrongPasswordData.csv`)
**Task**: Biometric Anomaly Detection (Genuine User vs Impostors)
**Target Passphrase**: `.tie5Roanl`
**Models**: Scaled Manhattan Baseline, One-Class SVM (RBF), Isolation Forest

Reference: Killourhy, K. S., & Maxion, R. A. (2009). *Comparing anomaly-detection algorithms for keystroke dynamics*. DSN 2009.

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.dataset import load_cmu_dataset, get_all_subjects, get_user_splits, FEATURE_COLUMNS
from ml.models import ScaledManhattanDetector, OneClassSVMDetector, IsolationForestDetector
from ml.evaluate import compute_eer, calculate_far_frr
from ml.benchmark import run_benchmark

print(f"Loaded {len(FEATURE_COLUMNS)} feature definitions.")

## 1. Load and Inspect CMU Benchmark Dataset
The dataset contains 51 subjects typing the password `.tie5Roanl` 400 times across 8 sessions (50 repetitions per session).

In [2]:
df = load_cmu_dataset("../data/DSL-StrongPasswordData.csv")
print("Dataset Shape:", df.shape)
print("Total Unique Subjects:", df["subject"].nunique())
df.head()

## 2. Feature Timing Distributions
Compare key hold times ($H$) and flight times ($DD$ / $UD$) for genuine vs impostor samples.

In [3]:
s002_train = df[df["subject"] == "s002"].iloc[:200]
plt.figure(figsize=(10, 4))
plt.boxplot([s002_train["H.period"], s002_train["H.t"], s002_train["H.five"], s002_train["H.Return"]],
            tick_labels=["H.period", "H.t", "H.five", "H.Return"])
plt.title("Key Hold Time Distributions for Subject s002 (Train Reps)")
plt.ylabel("Duration (seconds)")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Run Benchmark Across All 51 Subjects
Train each anomaly detector on the first 200 repetitions of each subject, and evaluate against the remaining 200 genuine reps and 250 impostor attempts.

In [4]:
summary = run_benchmark(
    data_path="../data/DSL-StrongPasswordData.csv",
    train_reps=200,
    impostors_per_other=5,
    output_dir="../results"
)

## 4. Benchmark Visualizations
Display ROC curves, DET curves, and model comparison bar chart.

In [5]:
from IPython.display import Image, display
display(Image(filename="../results/model_comparison.png"))
display(Image(filename="../results/roc_curves.png"))
display(Image(filename="../results/det_curves.png"))